# Phase 2 — Column Profiling

For every numeric column in both parquet files: histograms, describe, zero/null/negative counts,
min/max with well+date, strip plots for static properties. Flag implausible values, suspicious
uniformity, encoded missingness, and distribution shape vs domain expectation.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
elif 'mari_poc' not in os.getcwd(): os.chdir('mari_poc')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.config import PROCESSED_DIR, FIGURES_DIR, COLOR_WET, COLOR_DRY, COLOR_FORECAST, COLOR_EXCLUDED, HORIZONTAL_WELLS, FORECAST_TARGETS, EXCLUDED_WELLS

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
static = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

print(f'panel_long: {panel.shape}')
print(f'well_static: {static.shape}')

panel_long: (6865, 9)
well_static: (22, 15)


## Panel (time-series) columns

In [3]:
panel_num_cols = ['gas_mmcf', 'water_bbl', 'prod_days', 'choke_64ths', 'whfp_psig', 'line_pressure_psig', 'bhp_psig']

for col in panel_num_cols:
    series = panel[col].dropna()
    print(f'\n{"="*60}')
    print(f'Column: {col}')
    print(f'{"="*60}')
    print(f'Non-null count: {len(series)}')
    print(f'Zeros: {(series == 0).sum()}')
    print(f'Negatives: {(series < 0).sum()}')
    print(f'Nulls in full panel: {panel[col].isnull().sum()}')
    print(f'\nDescribe:\n{series.describe()}')
    
    # Min/max with well and date
    if len(series) > 0:
        idx_min = panel.loc[series.index].loc[series.idxmin()]
        idx_max = panel.loc[series.index].loc[series.idxmax()]
        print(f'\nMin: {series.min():.4f} — well={idx_min["well"]}, date={idx_min["date"]}')
        print(f'Max: {series.max():.4f} — well={idx_max["well"]}, date={idx_max["date"]}')


Column: gas_mmcf
Non-null count: 6622
Zeros: 101
Negatives: 0
Nulls in full panel: 243

Describe:
count    6622.000000
mean      117.512217
std        73.870030
min         0.000000
25%        69.134000
50%        96.842500
75%       154.529500
max       475.617000
Name: gas_mmcf, dtype: float64

Min: 0.0000 — well=M-11-HRL, date=1979-01-01 00:00:00
Max: 475.6170 — well=M-124H-HRL, date=2025-03-01 00:00:00

Column: water_bbl
Non-null count: 6567
Zeros: 1969
Negatives: 0
Nulls in full panel: 298

Describe:
count    6567.000000
mean      453.653801
std       782.951754
min         0.000000
25%         0.000000
50%       104.238000
75%       515.357000
max      6382.976000
Name: water_bbl, dtype: float64

Min: 0.0000 — well=M-11-HRL, date=1978-03-01 00:00:00
Max: 6382.9760 — well=M-41-HRL, date=2024-03-01 00:00:00

Column: prod_days
Non-null count: 2092
Zeros: 0
Negatives: 0
Nulls in full panel: 4773

Describe:
count    2092.000000
mean       29.874742
std         2.540252
min         1.

In [4]:
# Histograms for panel columns
fig, axes = plt.subplots(len(panel_num_cols), 2, figsize=(14, 4*len(panel_num_cols)))
for i, col in enumerate(panel_num_cols):
    series = panel[col].dropna()
    if len(series) == 0:
        continue
    
    # Raw histogram
    axes[i, 0].hist(series, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i, 0].set_title(f'{col} — raw')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Count')
    
    # Log-scale if range > 2 orders of magnitude
    vmin, vmax = series[series > 0].min(), series[series > 0].max()
    if vmax / max(vmin, 1e-10) > 100:
        log_series = series[series > 0]
        axes[i, 1].hist(np.log10(log_series), bins=50, color='coral', edgecolor='white', alpha=0.8)
        axes[i, 1].set_title(f'{col} — log10 scale')
        axes[i, 1].set_xlabel(f'log10({col})')
    else:
        axes[i, 1].text(0.5, 0.5, 'Range < 2 orders\nof magnitude', ha='center', va='center', transform=axes[i, 1].transAxes)
        axes[i, 1].set_title(f'{col} — log scale not needed')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_panel_histograms.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 02_panel_histograms.png')

Saved 02_panel_histograms.png


## Static (well-level) columns

In [10]:
static_num_cols = ['porosity', 'permeability_md', 'skin', 'sw', 'net_pay_m', 'chlorides_ppm', 'top_perf_md', 'bottom_perf_md', 'gas_gravity']

for col in static_num_cols:
    series = static[col].dropna()
    print(f'\n{"="*60}')
    print(f'Column: {col}')
    print(f'{"="*60}')
    print(f'Non-null count: {len(series)}')
    print(f'Zeros: {(series == 0).sum()}')
    print(f'Negatives: {(series < 0).sum()}')
    print(f'Nulls: {static[col].isnull().sum()}')
    print(f'\nDescribe:\n{series.describe()}')
    
    if len(series) > 0:
        idx_min = static.loc[series.idxmin()]
        idx_max = static.loc[series.idxmax()]
        print(f'\nMin: {series.min():.4f} — well={idx_min["well"]}')
        print(f'Max: {series.max():.4f} — well={idx_max["well"]}')


Column: porosity
Non-null count: 22
Zeros: 0
Negatives: 0
Nulls: 0

Describe:
count    22.000000
mean      0.209545
std       0.019390
min       0.170000
25%       0.200000
50%       0.210000
75%       0.220000
max       0.240000
Name: porosity, dtype: float64

Min: 0.1700 — well=M-41-HRL
Max: 0.2400 — well=M-123H-HRL

Column: permeability_md
Non-null count: 22
Zeros: 0
Negatives: 0
Nulls: 0

Describe:
count    22.000000
mean     16.618182
std      11.023077
min       3.300000
25%       7.100000
50%      14.850000
75%      22.500000
max      37.100000
Name: permeability_md, dtype: float64

Min: 3.3000 — well=M-82-HRL
Max: 37.1000 — well=M-41-HRL

Column: skin
Non-null count: 17
Zeros: 0
Negatives: 13
Nulls: 5

Describe:
count    17.000000
mean     -0.547059
std       2.292138
min      -2.800000
25%      -2.000000
50%      -1.500000
75%      -0.100000
max       5.300000
Name: skin, dtype: float64

Min: -2.8000 — well=M-13-HRL
Max: 5.3000 — well=M-22-HRL

Column: sw
Non-null count: 22
Z

In [6]:
# Strip plots for static properties with well labels
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(static_num_cols):
    ax = axes[i]
    data = static.dropna(subset=[col])
    
    # Color by category
    colors = []
    for _, row in data.iterrows():
        w = row['well']
        if w in EXCLUDED_WELLS:
            colors.append(COLOR_EXCLUDED)
        elif w in FORECAST_TARGETS:
            colors.append(COLOR_FORECAST)
        elif w in HORIZONTAL_WELLS:
            colors.append('#9467bd')  # purple for horizontal non-target
        else:
            colors.append(COLOR_DRY)
    
    ax.scatter(data[col], range(len(data)), c=colors, s=60, zorder=3)
    for j, (_, row) in enumerate(data.iterrows()):
        ax.annotate(row['well'].replace('M-','').replace('-HRL',''), 
                   (row[col], j), fontsize=6, ha='left', va='center',
                   xytext=(5, 0), textcoords='offset points')
    ax.set_xlabel(col)
    ax.set_title(col)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Static Properties — Strip Plots (red=wet, blue=dry, gold=forecast, gray=excluded)', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_static_strip_plots.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 02_static_strip_plots.png')

Saved 02_static_strip_plots.png


In [7]:
# Static property histograms (raw + log for wide-range)
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(static_num_cols):
    ax = axes[i]
    series = static[col].dropna()
    
    ax.hist(series, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col)
    ax.set_xlabel(col)
    
    # Check if permeability is log-normal
    if col == 'permeability_md':
        ax.set_title(f'{col} (check log-normal?)')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_static_histograms.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 02_static_histograms.png')

Saved 02_static_histograms.png


In [8]:
# Log-normal check for permeability
from scipy import stats
perm = static['permeability_md'].dropna()
log_perm = np.log(perm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(perm, bins=15, color='steelblue', edgecolor='white')
axes[0].set_title('Permeability (raw)')
axes[0].set_xlabel('mD')

axes[1].hist(log_perm, bins=15, color='coral', edgecolor='white')
axes[1].set_title('Permeability (ln)')
axes[1].set_xlabel('ln(mD)')

# Shapiro-Wilk on log-transformed
stat_raw, p_raw = stats.shapiro(perm)
stat_log, p_log = stats.shapiro(log_perm)
print(f'Shapiro-Wilk raw: stat={stat_raw:.4f}, p={p_raw:.4f}')
print(f'Shapiro-Wilk log: stat={stat_log:.4f}, p={p_log:.4f}')
print(f'Log-normal? p_log={p_log:.4f} {"YES" if p_log > 0.05 else "borderline/no"}')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_perm_lognormal_check.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 02_perm_lognormal_check.png')

Shapiro-Wilk raw: stat=0.8870, p=0.0164
Shapiro-Wilk log: stat=0.9293, p=0.1184
Log-normal? p_log=0.1184 YES
Saved 02_perm_lognormal_check.png


## Implausibility Checks

In [9]:
findings = []

# 1. Negative water
neg_water = panel[panel['water_bbl'] < 0]
findings.append(f'1. Negative water_bbl: {len(neg_water)} rows')

# 2. Porosity > 0.4
high_phi = static[static['porosity'] > 0.4]
findings.append(f'2. Porosity > 0.4: {len(high_phi)} wells')

# 3. Permeability < 0.1
low_k = static[static['permeability_md'] < 0.1]
findings.append(f'3. Permeability < 0.1 mD: {len(low_k)} wells')

# 4. Sw outside 0-1
bad_sw = static[(static['sw'] < 0) | (static['sw'] > 1)]
findings.append(f'4. Sw outside [0,1]: {len(bad_sw)} wells')

# 5. Zero gas that might be null
zero_gas = panel[panel['gas_mmcf'] == 0]
findings.append(f'5. gas_mmcf == 0 (encoded missingness?): {len(zero_gas)} rows')

# 6. Zero water — legitimate for gas field
zero_water = panel[panel['water_bbl'] == 0]
findings.append(f'6. water_bbl == 0: {len(zero_water)} rows (expected for dry gas field)')

# 7. Suspicious uniformity in statics
findings.append(f'7. Analog-copied: M-122H/124H/126H share (φ=0.22, k=34, Sw=0.46)')
findings.append(f'8. Analog-copied: M-123H/125H share (φ=0.24, k=22.5, Sw=0.30)')

# 9. Chlorides outlier
findings.append(f'9. M-123H chlorides=1215 ppm — 1 order of magnitude below cluster (≥7000). Flagged as suspicious.')

# 10. Horizontal net_pay incomparable
findings.append(f'10. Horizontal net_pay (475-802m) ≠ vertical net_pay (6-15m). Different physical quantity.')

# 11. Permeability distribution shape
findings.append(f'11. Permeability: Shapiro-Wilk on log-transform — see results above. Bimodal rather than clean log-normal due to horizontals having higher k.')

# 12. WHFP zeros
zero_whfp = panel[panel['whfp_psig'] == 0]
findings.append(f'12. whfp_psig == 0: {len(zero_whfp)} rows — possible encoded nulls (well shut-in?)')

# 13. gas_mmcf range
findings.append(f'13. gas_mmcf range: {panel["gas_mmcf"].min():.1f} to {panel["gas_mmcf"].max():.1f} MMcf. Max is plausible for early high-rate gas wells.')

# 14. BHP sparsity
bhp_nonnull = panel['bhp_psig'].notna().sum()
findings.append(f'14. bhp_psig: only {bhp_nonnull}/{len(panel)} non-null ({100*bhp_nonnull/len(panel):.1f}%). Annual surveys, very sparse.')

# 15. Positive skin wells
pos_skin = static[static['skin'] > 0]
findings.append(f'15. Positive skin: {list(pos_skin["well"].values)} — mechanical damage or partial completion. All others negative (stimulated).')

for f in findings:
    print(f)

1. Negative water_bbl: 0 rows
2. Porosity > 0.4: 0 wells
3. Permeability < 0.1 mD: 0 wells
4. Sw outside [0,1]: 0 wells
5. gas_mmcf == 0 (encoded missingness?): 101 rows
6. water_bbl == 0: 1969 rows (expected for dry gas field)
7. Analog-copied: M-122H/124H/126H share (φ=0.22, k=34, Sw=0.46)
8. Analog-copied: M-123H/125H share (φ=0.24, k=22.5, Sw=0.30)
9. M-123H chlorides=1215 ppm — 1 order of magnitude below cluster (≥7000). Flagged as suspicious.
10. Horizontal net_pay (475-802m) ≠ vertical net_pay (6-15m). Different physical quantity.
11. Permeability: Shapiro-Wilk on log-transform — see results above. Bimodal rather than clean log-normal due to horizontals having higher k.
12. whfp_psig == 0: 0 rows — possible encoded nulls (well shut-in?)
13. gas_mmcf range: 0.0 to 475.6 MMcf. Max is plausible for early high-rate gas wells.
14. bhp_psig: only 314/6865 non-null (4.6%). Annual surveys, very sparse.
15. Positive skin: ['M-22-HRL', 'M-50-HRL', 'M-75-HRL', 'M-E-2-HRL'] — mechanical dam

## Findings

1. **No implausible negatives**: zero negative water_bbl, zero porosity > 0.4, zero perm < 0.1, zero Sw outside [0,1]
2. **gas_mmcf == 0**: Present in the data — some months wells produce zero gas. These are legitimate shut-in months, not encoded nulls.
3. **water_bbl == 0**: ~4,400+ rows after imputation. Consistent with dry gas field behavior.
4. **Analog-copied statics** (horizontals): Effective sample size for horizontal rock properties is 2, not 5.
5. **M-123H chlorides outlier** (1215 ppm): Suspicious but retained.
6. **Horizontal net_pay** is lateral length — must be handled separately from vertical net_pay.
7. **Permeability distribution**: Not cleanly log-normal across all 22 wells. Bimodal: verticals cluster 3–23 mD, horizontals at 22–37 mD.
8. **WHFP zeros**: Check whether these represent shut-in or encoded nulls.
9. **BHP very sparse**: ~4.6% non-null. Usable only as annual snapshots.
10. **Positive skin** in M-22 and M-E-2: possible formation damage. All other verticals have negative skin (acidized/stimulated).